<a href="https://colab.research.google.com/github/adildafedar/Natural-Language-Processing/blob/main/Assignment_%244.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Develop a Python script to perform part-of-speech tagging on a short text dataset.


In [9]:
import numpy as np
import re
from collections import Counter

In [10]:
corpus = """
king is a strong man
queen is a wise woman
boy is a young man
girl is a young woman
"""

embedding_dim = 8   # D
window_size = 2     # context size
epochs = 150
learning_rate = 0.01

In [11]:
import re

def preprocess(text):
    text = text.lower()                           # lowercase
    text = re.sub(r'[^a-z\s]', '', text)          # remove punctuation
    words = text.split()                          # tokenize
    return words

words = preprocess(corpus)
print(words)

['king', 'is', 'a', 'strong', 'man', 'queen', 'is', 'a', 'wise', 'woman', 'boy', 'is', 'a', 'young', 'man', 'girl', 'is', 'a', 'young', 'woman']


In [12]:
from collections import Counter

vocab = list(set(words))                          # unique words
V = len(vocab)

word_to_index = {w: i for i, w in enumerate(vocab)}
index_to_word = {i: w for w, i in word_to_index.items()}

print("Vocabulary:", vocab)
print("Vocab size:", V)

Vocabulary: ['strong', 'boy', 'a', 'man', 'king', 'girl', 'woman', 'young', 'is', 'queen', 'wise']
Vocab size: 11


In [13]:
training_data = []

for i, word in enumerate(words):
    target = word_to_index[word]

    for j in range(-window_size, window_size + 1):
        if j != 0 and 0 <= i + j < len(words):
            context_word = words[i + j]
            context = word_to_index[context_word]
            training_data.append((target, context))

print("Sample pairs:", training_data[:10])

Sample pairs: [(4, 8), (4, 2), (8, 4), (8, 2), (8, 0), (2, 4), (2, 8), (2, 0), (2, 3), (0, 8)]


In [14]:
# Input weights (V x D)
W1 = np.random.randn(V, embedding_dim)

# Output weights (D x V)
W2 = np.random.randn(embedding_dim, V)

In [15]:
def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / np.sum(e_x)

In [16]:
for epoch in range(epochs):
    total_loss = 0

    for target, context in training_data:

        # Forward pass
        h = W1[target]                 # (D,)
        u = np.dot(h, W2)              # (V,)
        y_pred = softmax(u)

        # Loss
        loss = -np.log(y_pred[context])
        total_loss += loss

        # Backprop
        y_true = np.zeros(V)
        y_true[context] = 1

        error = y_pred - y_true

        dW2 = np.outer(h, error)       # (D x V)
        dW1 = np.dot(W2, error)        # (D,)

        # Update
        W1[target] -= learning_rate * dW1
        W2 -= learning_rate * dW2

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.4f}")

Epoch 0, Loss: 339.9459
Epoch 50, Loss: 145.7442
Epoch 100, Loss: 140.2126


In [17]:
# Each row of W1 is a word embedding
def get_embedding(word):
    return W1[word_to_index[word]]

print("Embedding for 'king':", get_embedding("king"))

Embedding for 'king': [ 0.38464216 -1.91541041 -0.20955801 -0.36289647 -0.91380317  0.94875928
 -0.40543539 -0.21875043]


In [18]:
def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

w1 = "king"
w2 = "queen"

sim = cosine_similarity(get_embedding(w1), get_embedding(w2))
print(f"Similarity between {w1} and {w2}:", sim)

Similarity between king and queen: 0.5633437795927738


In [19]:
def most_similar(word):
    target_vec = get_embedding(word)
    sims = {}

    for w in vocab:
        if w != word:
            sims[w] = cosine_similarity(target_vec, get_embedding(w))

    return sorted(sims.items(), key=lambda x: x[1], reverse=True)

print("Most similar to 'king':", most_similar("king")[:5])

Most similar to 'king': [('wise', np.float64(0.612048074208519)), ('queen', np.float64(0.5633437795927738)), ('man', np.float64(0.3619502900481551)), ('girl', np.float64(0.34960890872569983)), ('young', np.float64(0.25359491329978473))]


In [20]:
print("=== WORD EMBEDDINGS ===\n")

for word in vocab:
    print(f"{word:10s} -> {get_embedding(word)}")

=== WORD EMBEDDINGS ===

strong     -> [ 0.56614614  1.02886006 -0.08736647 -0.2005009  -1.02054915  0.42834929
 -0.0351752  -0.74608176]
boy        -> [ 0.61291605  1.43333856  1.32399008  0.12258604 -1.59443769  1.15123315
 -0.92390262  1.19705537]
a          -> [ 0.9185268  -0.3460534   0.33956115 -0.31911652  0.28760171 -0.16587073
  0.03426633  0.61801481]
man        -> [ 0.06434557 -0.59326912  0.1549789  -1.77820038  0.8706053   1.11888648
 -0.08303714 -0.02925381]
king       -> [ 0.38464216 -1.91541041 -0.20955801 -0.36289647 -0.91380317  0.94875928
 -0.40543539 -0.21875043]
girl       -> [ 1.71682785 -0.47632316 -0.50697413 -0.41738242  0.74715924  0.39592436
 -1.69015145 -0.84124276]
woman      -> [ 0.03255659  0.66818532 -0.71197918 -0.05922165 -0.60603977  0.34613192
 -0.86073307  0.66993027]
young      -> [ 0.54250873  0.0591504   0.83503799 -0.63988345 -0.51584378  0.34836607
 -0.37700355 -2.13356426]
is         -> [-0.91418206 -0.77669304 -0.54263723  0.75344616  0.32023

In [21]:
print("\n=== PAIRWISE COSINE SIMILARITY ===\n")

for i, w1 in enumerate(vocab):
    for j, w2 in enumerate(vocab):
        if i < j:  # avoid duplicates
            sim = cosine_similarity(get_embedding(w1), get_embedding(w2))
            print(f"{w1:10s} - {w2:10s} : {sim:.4f}")


=== PAIRWISE COSINE SIMILARITY ===

strong     - boy        : 0.5102
strong     - a          : -0.2712
strong     - man        : -0.1457
strong     - king       : -0.0333
strong     - girl       : 0.1399
strong     - woman      : 0.3704
strong     - young      : 0.5926
strong     - is         : -0.3545
strong     - queen      : 0.2549
strong     - wise       : -0.0406
boy        - a          : 0.1288
boy        - man        : -0.1163
boy        - king       : -0.0216
boy        - girl       : -0.0591
boy        - woman      : 0.5725
boy        - young      : 0.0564
boy        - is         : -0.6222
boy        - queen      : -0.3007
boy        - wise       : 0.3148
a          - man        : 0.3043
a          - king       : 0.1559
a          - girl       : 0.3507
a          - woman      : -0.1292
a          - young      : -0.1736
a          - is         : -0.6193
a          - queen      : -0.4128
a          - wise       : 0.0540
man        - king       : 0.3620
man        - girl       :